# 18 · Running it for real

Everything so far you started by running a cell. In production nobody is awake
to run a cell.

This notebook is the deployment: three containers, one clock, and a break that
nobody triggers by hand.

In [ ]:
import sys; sys.path.insert(0, '..')
import subprocess, httpx, os, time
from pipelines.lib.config import dsn, SCHEMA

def compose(*args):
    r = subprocess.run(['docker', 'compose', '-f', '../platform/docker-compose.yml', *args],
                       capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

print(compose('ps', '--format', '{{.Name}}\t{{.Status}}'))

---

## Three containers, not two

The signal board is genuinely two things, and they are separated on purpose.

| Container | Runs | Why separate |
|---|---|---|
| `signal-api` | `uvicorn signal_service.api:app` | answers questions |
| `signal-scheduler` | `python -m signal_service.scheduler` | asks them on a clock |
| `agent` | `uvicorn agent_service.worker:app` | asleep until a record arrives |

A dashboard hammering `/signals` must not be able to slow the clock down, and a
slow clock must not make the dashboard time out. They share a package and share
nothing else.

## Start them

In [ ]:
print(compose('up', '-d', 'signal-api', 'signal-scheduler', 'agent'))

In [ ]:
SIGNAL = os.environ.get('SIGNAL_SERVICE_URL', 'http://localhost:8091')
AGENT  = os.environ.get('AGENT_SERVICE_URL',  'http://localhost:8092')

for name, url in [('signal', SIGNAL), ('agent', AGENT)]:
    for _ in range(30):
        try:
            print(f'{name:7}', httpx.get(f'{url}/health', timeout=5).json()); break
        except Exception:
            time.sleep(2)
    else:
        print(f'{name:7} not reachable at {url}')

### Where the secrets are

Look at the compose file: `env_file: [../.env]`. The model key and the Atlassian
token come from `.env`, which is git ignored. **No secret is written into a file
that gets committed**, and the same file works for a laptop and a container.

And note the environment block: `POSTGRES_HOST: postgres`, not `localhost`.
`config.py` loads `.env` with `override=False`, so the container's value wins
and not one line of service code changes between the two.

---

## Watch the clock work

In [ ]:
print(compose('logs', '--tail', '12', 'signal-scheduler'))

---

## Now break something, and touch nothing else

This is the whole point of the class. From here on, nobody types anything.

In [ ]:
before = httpx.get(f'{SIGNAL}/signals', timeout=120).json()
print(f'before: {before["breached"]} breached')
for s in before['signals']:
    if s['breached']:
        print(f'   {s["kpi"]:26} {s["value"]:,.2f}{s["unit"]}')

In [ ]:
run = subprocess.run([sys.executable, '../break_it.py'], capture_output=True, text=True,
                     cwd='..')
print(run.stdout + run.stderr)

for mod in ('pipelines.p3_bronze_driver_app', 'pipelines.p7_silver_rides',
            'pipelines.p8_gold_daily'):
    subprocess.run([sys.executable, '-m', mod], capture_output=True, cwd='..')
print('pipelines re-run. Every one of them succeeded.')

**Nothing failed.** No pipeline errored, no row count changed, and the warehouse
looks healthy from every angle except one.

Now wait for the clock. It runs every sixty seconds by default.

In [ ]:
import psycopg

seen = None
for i in range(20):
    with psycopg.connect(dsn()) as c:
        row = c.execute(f"""SELECT breach_id, kpi, status FROM oncall.breaches
                           WHERE kpi = 'surge_coverage_pct'
                           ORDER BY detected_at DESC LIMIT 1""").fetchone()
    if row:
        seen = row
        print(f'  {i*10:>4}s  the board raised {row[0]}  ({row[2]})')
        break
    print(f'  {i*10:>4}s  nothing yet')
    time.sleep(10)

## And the agent picked it up without being asked

In [ ]:
for i in range(30):
    r = httpx.get(f'{AGENT}/investigations/{seen[0]}', timeout=30)
    if r.status_code == 200:
        d = r.json()
        print(f'  {i*10:>4}s  {d["status"]:20} {d.get("steps") or ""}')
        if d['status'] in ('done', 'waiting_for_human', 'failed'):
            break
    else:
        print(f'  {i*10:>4}s  not started')
    time.sleep(10)

In [ ]:
print(d.get('handover') or d.get('error'))
print('\nARTIFACTS')
for p in d['pages']:            print('   page   ', p['url'] or p['page_id'])
for t in d['tickets']:          print('   ticket ', t['ticket_id'], t['kind'], t['severity'])
for c in d['change_requests']:  print('   change ', c['request_id'], c['kind'], c['status'])

---

## What just happened

1. A field moved upstream. Nothing failed
2. The clock evaluated thirteen KPIs, as it does every minute
3. One breached, and the board **wrote a record, then rang a doorbell**
4. The agent service accepted it and returned immediately, so the clock kept ticking
5. Five agents investigated, in order, stopping when they had enough
6. A page and a ticket appeared, with an owner and a diagram

**Nobody typed anything after step 1.**

## Put it back

In [ ]:
print(subprocess.run([sys.executable, '../break_it.py', '--fix'],
                    capture_output=True, text=True, cwd='..').stdout)
for mod in ('pipelines.p7_silver_rides', 'pipelines.p8_gold_daily'):
    subprocess.run([sys.executable, '-m', mod], capture_output=True, cwd='..')
print(httpx.post(f'{SIGNAL}/evaluate', timeout=180).json()['breached'], 'still breached')

---

## What you learned

- **Three containers**, because answering questions and asking them are different jobs
- Secrets come from `.env` through `env_file`, never from a committed file
- `override=False` is why `localhost` and `postgres` are the same code
- The doorbell **returns immediately**, so a two minute investigation cannot stall
  a sixty second clock
- The record is durable **before** the doorbell rings, and a sweep is the backstop
- From a field moving to a page with an owner on it: **nobody typed anything**